In [1]:
from pyspark.sql.functions import (
    col, trim, upper, when, current_timestamp,
    lit, to_date, sha2
)
from pyspark.sql.types import DecimalType, IntegerType
from datetime import datetime

BRONZE_TABLE  = "Interac_Bronze.dbo.cardholders"
SILVER_TABLE  = "silver_cardholders"
SILVER_DB     = "Interac_Fabric_Workspace.Interac_Silver.dbo"
PIPELINE_NAME = "NB_04_Silver_Cardholders"
BATCH_DATE    = datetime.now().strftime("%Y-%m-%d")

print(f"Silver Cardholders Pipeline")
print(f"Started: {datetime.now()}")

StatementMeta(, 1bc1e2e8-e530-4cee-95d9-289d6efbada6, 3, Finished, Available, Finished, False)

Silver Cardholders Pipeline
Started: 2026-05-06 00:21:38.739230


In [2]:
df_bronze = spark.read.table(BRONZE_TABLE)
total_bronze = df_bronze.count()
print(f"Bronze rows read: {total_bronze:,}")

dq_results = {}
dq_results["null_cardholder_id"] = df_bronze.filter(col("cardholder_id").isNull()).count()
dq_results["null_province"] = df_bronze.filter(col("province").isNull()).count()
dq_results["blocked_cardholders"] = df_bronze.filter(col("status") == "BLOCKED").count()
dq_results["inactive_cardholders"] = df_bronze.filter(col("status") == "INACTIVE").count()
dq_results["deceased_cardholders"] = df_bronze.filter(col("status") == "DECEASED").count()
dq_results["high_risk_cardholders"] = df_bronze.filter(col("risk_rating") == "HIGH").count()
dq_results["no_2fa"] = df_bronze.filter(col("two_factor_method") == "NONE").count()
dq_results["low_credit_score"] = df_bronze.filter(col("credit_score") < 600).count()

print("\nDQ CHECK RESULTS:")
print("-" * 45)
for check, count_val in dq_results.items():
    status = "⚠ FLAGGED" if count_val > 0 else "✓ PASSED"
    print(f"{check:<35} {count_val:>6,}  {status}")

StatementMeta(, 1bc1e2e8-e530-4cee-95d9-289d6efbada6, 4, Finished, Available, Finished, False)

Bronze rows read: 5,000

DQ CHECK RESULTS:
---------------------------------------------
null_cardholder_id                       0  ✓ PASSED
null_province                            0  ✓ PASSED
blocked_cardholders                    160  ⚠ FLAGGED
inactive_cardholders                   259  ⚠ FLAGGED
deceased_cardholders                    45  ⚠ FLAGGED
high_risk_cardholders                  325  ⚠ FLAGGED
no_2fa                                 258  ⚠ FLAGGED
low_credit_score                       317  ⚠ FLAGGED


In [3]:
df_quarantine = df_bronze.filter(
    col("cardholder_id").isNull() | col("province").isNull()
)
quarantine_count = df_quarantine.count()

if quarantine_count > 0:
    (df_quarantine
        .withColumn("_quarantine_reason", lit("NULL_PRIMARY_KEY_OR_PROVINCE"))
        .withColumn("_quarantined_at", current_timestamp())
        .write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER_DB}.silver_cardholders_quarantine"))
    print(f"Quarantined: {quarantine_count:,} records")

df_valid = df_bronze.filter(
    col("cardholder_id").isNotNull() & col("province").isNotNull()
)

df_silver = (df_valid
    .withColumn("cardholder_id",    trim(col("cardholder_id")))
    .withColumn("first_name",       trim(col("first_name")))
    .withColumn("last_name",        trim(col("last_name")))
    .withColumn("province",         upper(trim(col("province"))))
    .withColumn("issuing_bank",     trim(col("issuing_bank")))
    .withColumn("risk_rating",      upper(trim(col("risk_rating"))))
    .withColumn("status",           upper(trim(col("status"))))
    .withColumn("income_band",      upper(trim(col("income_band"))))
    .withColumn("two_factor_method",upper(trim(col("two_factor_method"))))
    .withColumn("first_name_masked", sha2(col("first_name"), 256))
    .withColumn("last_name_masked",  sha2(col("last_name"), 256))
    .withColumn("dob_masked",
        sha2(col("date_of_birth").cast("string"), 256))
    .withColumn("postal_code_masked", sha2(col("postal_code"), 256))
    .drop("first_name", "last_name", "date_of_birth", "postal_code")
    .withColumn("registration_date",
        to_date(col("registration_date"), "yyyy-MM-dd"))
    .withColumn("last_login_date",
        to_date(col("last_login_date"), "yyyy-MM-dd"))
    .withColumn("last_modified_date",
        to_date(col("last_modified_date"), "yyyy-MM-dd"))
    .withColumn("credit_score",
        col("credit_score").cast(IntegerType()))
    .withColumn("interac_daily_limit_cad",
        col("interac_daily_limit_cad").cast(IntegerType()))
    .withColumn("etransfer_monthly_limit_cad",
        col("etransfer_monthly_limit_cad").cast(IntegerType()))
    .withColumn("is_active",
        when(col("status") == "ACTIVE", "Y").otherwise("N"))
    .withColumn("is_high_risk",
        when(col("risk_rating") == "HIGH", "Y").otherwise("N"))
    .withColumn("has_2fa",
        when(col("two_factor_method") == "NONE", "N").otherwise("Y"))
    .withColumn("credit_band",
        when(col("credit_score") < 620, "POOR")
        .when(col("credit_score") < 680, "FAIR")
        .when(col("credit_score") < 740, "GOOD")
        .when(col("credit_score") < 800, "VERY_GOOD")
        .otherwise("EXCELLENT"))
    .withColumn("_silver_loaded_at", current_timestamp())
    .withColumn("_pipeline_name",    lit(PIPELINE_NAME))
    .withColumn("_batch_date",       lit(BATCH_DATE))
    .withColumn("_is_current",       lit("Y"))
    .drop("_ingested_at", "_source_file", "_lakehouse")
)

print(f"Valid records: {df_silver.count():,}")

StatementMeta(, 1bc1e2e8-e530-4cee-95d9-289d6efbada6, 5, Finished, Available, Finished, False)

Valid records: 5,000


In [4]:
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .saveAsTable(f"{SILVER_DB}.{SILVER_TABLE}"))

spark.sql(f"OPTIMIZE {SILVER_DB}.{SILVER_TABLE} ZORDER BY (cardholder_id, province)")

final_count = spark.read.table(f"{SILVER_DB}.{SILVER_TABLE}").count()

print("\n" + "="*60)
print("SILVER CARDHOLDERS SUMMARY")
print("="*60)
print(f"Bronze rows in    : {total_bronze:,}")
print(f"Quarantined       : {quarantine_count:,}")
print(f"Silver rows out   : {final_count:,}")
print(f"Pass rate         : {round(final_count/total_bronze*100, 2)}%")
print(f"PII masked        : first_name, last_name, date_of_birth, postal_code")
print(f"Table             : {SILVER_DB}.{SILVER_TABLE}")
print(f"Completed at      : {datetime.now()}")
print("="*60)

StatementMeta(, 1bc1e2e8-e530-4cee-95d9-289d6efbada6, 6, Finished, Available, Finished, True)


SILVER CARDHOLDERS SUMMARY
Bronze rows in    : 5,000
Quarantined       : 0
Silver rows out   : 5,000
Pass rate         : 100.0%
PII masked        : first_name, last_name, date_of_birth, postal_code
Table             : Interac_Fabric_Workspace.Interac_Silver.dbo.silver_cardholders
Completed at      : 2026-05-06 00:22:33.140609
